# Data Cleaning

In [1]:
import pandas as pd

## Data Loading

In [2]:
# Load the CSV file
df = pd.read_csv('C:\\Users\\frank\\data_science_project_advance\\datasets\\unclean_sales.csv')

## Expolration 

### Inspecting the data

In [3]:
# Check the first few rows
print(df.head())

# Check column data types and missing values
print(df.info())

# Get summary statistics
print(df.describe(include='all'))


         Date    Product Region   Sales
0  2023-01-09   Widget A   West  1143.0
1  2023-01-16   Widget A  North  1974.0
2  2023-02-01   Widget C   East  1825.0
3  2023-01-05   widget C   eest     NaN
4  2023-02-09  Widget A   SOUTH  4766.0
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1995 entries, 0 to 1994
Data columns (total 4 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   Date     1995 non-null   object 
 1   Product  1995 non-null   object 
 2   Region   1995 non-null   object 
 3   Sales    1892 non-null   float64
dtypes: float64(1), object(3)
memory usage: 62.5+ KB
None
              Date   Product  Region         Sales
count         1995      1995    1995   1892.000000
unique          61         8       8           NaN
top     2023-02-25  Widget A  north            NaN
freq            45       267     273           NaN
mean           NaN       NaN     NaN   2984.247886
std            NaN       NaN     NaN   2293.311484
min        

###  Convert Date Column to datetime

In [4]:
df['Date'] = pd.to_datetime(df['Date'], errors='coerce')

### Check for Missing Values

In [5]:
print(df.isnull().sum())

Date         0
Product      0
Region       0
Sales      103
dtype: int64


### Handle Missing Values in Sales
There are 103 missing sales values (1995 - 1892 = 103)

#### Drop rows with missing sales


In [6]:
df = df.dropna(subset=['Sales'])

### Remove Duplicates (if any)

In [7]:
df = df.drop_duplicates()

### Check for Inconsistent or Unexpected Categories

In [8]:
print(df['Product'].value_counts())
print(df['Region'].value_counts())

Product
Widget A     240
widget a     230
Widgget B    229
Widget A     228
Widget D     228
widget C     222
Widget B     218
Widget C     208
Name: count, dtype: int64
Region
north     244
South     242
West      230
North     228
SOUTH     218
eest      215
East      214
West      212
Name: count, dtype: int64


#### Fix typos or inconsistencies if needed

In [9]:
df['Region'] = df['Region'].str.strip().str.title()  # Normalize case and remove extra spaces

### Handle Outliers in Sales 

#### removing or flaging outliers:

In [10]:
# Example: Remove sales above 99th percentile
upper_limit = df['Sales'].quantile(0.99)
df = df[df['Sales'] <= upper_limit]

### Final Checks

In [11]:
print(df.info())
print(df.describe())

<class 'pandas.core.frame.DataFrame'>
Index: 1784 entries, 0 to 1994
Data columns (total 4 columns):
 #   Column   Non-Null Count  Dtype         
---  ------   --------------  -----         
 0   Date     1784 non-null   datetime64[ns]
 1   Product  1784 non-null   object        
 2   Region   1784 non-null   object        
 3   Sales    1784 non-null   float64       
dtypes: datetime64[ns](1), float64(1), object(2)
memory usage: 69.7+ KB
None
                                Date         Sales
count                           1784   1784.000000
mean   2023-01-31 09:58:55.426008832   2791.195067
min              2023-01-01 00:00:00    502.000000
25%              2023-01-16 00:00:00   1699.000000
50%              2023-01-31 12:00:00   2852.000000
75%              2023-02-16 00:00:00   3873.250000
max              2023-03-02 00:00:00  11777.000000
std                              NaN   1278.353182


### Find Duplicates Based on Product and Region

In [12]:
# Check for duplicates based on Product and Region only
duplicates = df[df.duplicated(subset=['Product', 'Region'], keep=False)]
print(duplicates)

           Date    Product Region   Sales
0    2023-01-09   Widget A   West  1143.0
1    2023-01-16   Widget A  North  1974.0
2    2023-02-01   Widget C   East  1825.0
4    2023-02-09  Widget A   South  4766.0
6    2023-01-25   widget a  South   658.0
...         ...        ...    ...     ...
1990 2023-01-21   Widget B   Eest   771.0
1991 2023-01-07   Widget D  South  2313.0
1992 2023-01-11   Widget D  South  3012.0
1993 2023-02-18   Widget C   East  2173.0
1994 2023-01-22   Widget D  North  4808.0

[1784 rows x 4 columns]


### Remove Duplicates Based on Product and Region

#### This will keep the first occurrence and remove the rest: 

In [13]:
df = df.drop_duplicates(subset=['Product', 'Region'], keep='first')

### Example: Sum of Sales

In [14]:
df = df.groupby(['Product', 'Region'], as_index=False)['Sales'].sum()

### Example: Average Sales

In [15]:
df = df.groupby(['Product', 'Region'], as_index=False)['Sales'].mean()

### Re-check for Duplicates

In [16]:
print(df.duplicated(subset=['Product', 'Region']).sum())  # Should return 0

0


### Fix Region Misspelling

In [17]:
df['Region'] = df['Region'].replace('Eest', 'East')
df['Product'] = df['Product'].replace(['widgget', 'Widgget', 'widget'], 'Widget')
df['Product'] = df['Product'].replace('widget a', 'Widget A')
df['product'] = df['Product'].replace('Widgget B', 'Widget B')


### Normalize Capitalization and Spacing

In [18]:
# Remove leading/trailing spaces and standardize title case (e.g., "widget a" → "Widget A")
df['Product'] = df['Product'].str.strip().str.title()
df['Region'] = df['Region'].str.strip().str.title()


### Confirm Fixing

In [19]:
print(df['Product'].unique())
print(df['Region'].unique())

['Widget A' 'Widget B' 'Widget C' 'Widget D' 'Widgget B']
['East' 'North' 'South' 'West']


In [22]:
# Find duplicate products
duplicates = df[df.duplicated(subset=['Product'], keep=False)]
print(duplicates)

      Product Region   Sales    product
0    Widget A   East  4508.0   Widget A
1    Widget A   East  1145.0   Widget A
2    Widget A  North  1974.0   Widget A
3    Widget A  South  3773.0   Widget A
4    Widget A   West  1143.0   Widget A
5    Widget A   East  4933.0  Widget A 
6    Widget A   East  3474.0  Widget A 
7    Widget A  North  3722.0  Widget A 
8    Widget A  South  4766.0  Widget A 
9    Widget A   West  4129.0  Widget A 
10   Widget B   East   653.0   Widget B
11   Widget B   East  1764.0   Widget B
12   Widget B  North  4270.0   Widget B
13   Widget B  South  2742.0   Widget B
14   Widget B   West  1966.0   Widget B
15   Widget C   East  1825.0   Widget C
16   Widget C   East   506.0   Widget C
17   Widget C  North  2867.0   Widget C
18   Widget C  South  3921.0   Widget C
19   Widget C   West  2894.0   Widget C
20   Widget D   East  2954.0   Widget D
21   Widget D   East  2174.0   Widget D
22   Widget D  North  1487.0   Widget D
23   Widget D  South  2312.0   Widget D


In [24]:
df = df.drop_duplicates(subset='Product', keep='first')
df = df.sort_values('Sales', ascending=False).drop_duplicates(subset='Product', keep='first')

In [25]:
print(df.duplicated(subset='Product').sum())  # Should return 0

0


In [29]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
Index: 5 entries, 0 to 10
Data columns (total 4 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   Product  5 non-null      object 
 1   Region   5 non-null      object 
 2   Sales    5 non-null      float64
 3   product  5 non-null      object 
dtypes: float64(1), object(3)
memory usage: 200.0+ bytes
None


### Save cleaned data

In [26]:
df.to_csv("C:\\Users\\frank\\data_science_project_advance\\datasets\\cleaned_data.csv", index=False)